# Figure S4 - dendrograms


In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)

remove_UA_RU=False
smooth=False


In [ ]:
DFfreq = pd.read_csv(r'pivotUkraine_SUM_excelInput_weekly.csv')
#Load frequency share pivoted df
DFfreq.set_index(DFfreq.columns[0], inplace=True)
DFfreq.head(3)

In [ ]:
manual_order = [
     'Ukrainian', 'Russian', 'Arabic', 'Portuguese', 'Catalan', 'Korean', 'Persian', 'Turkish', 'Indonesian', 'Urdu', 'Vietnamese',
    'Serbian', 'Estonian', 'Romanian', 'Greek', 'Hungarian', 'Polish', 'Swedish', 'Czech', 'Spanish', 
    'Danish', 'English', 'Dutch', 'Norwegian','Finnish', 'French', 'Italian', 'German'
]

manual_order.reverse()

# otherwise, to respect manual_order but keep any extras at the end (or as NaN rows):
DFfreq = DFfreq.reindex(manual_order)

DFfreqLog = DFfreq.applymap(lambda x: 0 if x == 0 else (np.log10(x * 100000) if x > 0 else -np.log10(-x * 100000)))
DFfreq

# Find overexpressions etc

In [ ]:
# WITHIN LANGUAGE
if remove_UA_RU==True:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'])

          ###### 1-Mean freq of language
#What is the average freq of each language>?
freqRowMeans = DFfreq.mean(axis=1)

          ###### 2-Over and underexplression
#Mean per language
WLDFresultNonLog = DFfreq.sub(freqRowMeans, axis=0)
WLDFresultNonLog = WLDFresultNonLog.div(freqRowMeans, axis=0)

          ###### 3-Log Over and underexplression
#Put the last result to log scale (take into account neg values)
WLDFresultLog = WLDFresultNonLog.applymap(lambda x: 0 if x == 0 else (np.log10(x * 100000) if x > 0 else -np.log10(-x * 100000)))

#Smooth data
smooth_time=6
min_periods=1
if smooth:
    WLDFresultLog = WLDFresultLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    WLDFresultNonLog = WLDFresultNonLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    #DFfreq = DFfreq.T.rolling(window=smooth_time, min_periods=min_periods).mean().T


In [ ]:
# BETWEEN LANGUAGES

if remove_UA_RU==True:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'])

          ###### 1-FreqShare
#Total fraction of usage for each week (all languages together)
SumsFreq = DFfreq.sum()

          ###### 2-LangShare-in-TotalFreqShare
#get each nonlog value share from the sum of values of that time unit
DFfreqShare = DFfreq.div(SumsFreq, axis=1)

          ###### 3-LangShareInTot-Expression
#What is the average freq of each language>?
#Avg of each row of row of freqshare (raw frequencies avg per language)
freqRowMeans = DFfreq.mean(axis=1)

#Based on average of each language, how big share of total 100% attention does each language have?
#Mean share of frequency for each langauge in relation to all languages mean frequenceis
freqShareRowMeans_Proportion = freqRowMeans / freqRowMeans.sum()

#Now that we know the expected share in average and share for each time point how much do they differ?
#What is the difference when subtracting from each DFfreqShare weekly share the average share of freqShareRowMeans_Pro...
BLDFresultNonLog = DFfreqShare.sub(freqShareRowMeans_Proportion, axis=0)

#Put the last result to log scale (take into account neg values)
BLDFresultLog = BLDFresultNonLog.applymap(lambda x: 0 if x == 0 else (np.log10(x * 100000) if x > 0 else -np.log10(-x * 100000)))

#Smooth data
smooth_time=6
min_periods=1
if smooth:
    BLDFresultLog = BLDFresultLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    BLDFresultNonLog = BLDFresultNonLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    #DFfreq = DFfreq.T.rolling(window=smooth_time, min_periods=min_periods).mean().T


# Dendograms

In [ ]:
directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Visualizations\Fig.S5_dendograms'
os.chdir(directory)


In [ ]:
#RAW TEST
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from datetime import datetime
date_str = datetime.now().strftime("%Y-%m-%d_%H.%M.%S")


# Compute the hierarchical clustering using Ward's method
Z = linkage(WLDFresultLog, method='ward')

# Create the plot
plt.figure(figsize=(3.46457, 1))
dendrogram(Z, labels=WLDFresultLog.index, leaf_rotation=90)
#plt.title("Dendrogram of Languages based on Expression Across Time")
#plt.xlabel("Language")
plt.ylabel("Distance")
plt.tight_layout()

# reset x/y tick‐label font sizes
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)

# Save the dendrogram as a PNG file
#plt.savefig(f"dendrogram_{date_str}.png", format="png", dpi=500)

# Optionally, display the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from datetime import datetime

def save_dendrogram(df, name, method='ward', dpi=500, figsize=(7, 3)):
    """
    Compute and save a dendrogram for any DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        Rows are the observations to cluster; the index labels are used on the leaves.
    name : str
        A short name for the DataFrame to include in the output filename.
    method : str, optional
        Linkage method for clustering (default 'ward').
    dpi : int, optional
        Resolution for the saved PNG (default 500).
    figsize : tuple, optional
        Figure size in inches (default (9, 4)).

    Returns
    -------
    filenames : dict
        Filenames of the saved figures {'png': ..., 'svg': ...}
    """
    # build date string
    date_str = datetime.now().strftime("%Y-%m-%d")

    # compute linkage
    Z = linkage(df, method=method)

    # plot dendrogram
    plt.figure(figsize=figsize)
    dendrogram(Z, labels=df.index, leaf_rotation=90)
    plt.ylabel("Distance")
    plt.tight_layout()

    # base filename
    base = f"dendrogram_{name}_{date_str}"

    # save PNG
    png_file = f"{base}.png"
    plt.savefig(png_file, format="png", dpi=dpi)

    # save SVG (vector format)
    svg_file = f"{base}.svg"
    plt.savefig(svg_file, format="svg")

    plt.show()

    return {"png": png_file, "svg": svg_file}


In [ ]:
#save_dendrogram(WLDFresultLog,'WL_Log')


In [ ]:
save_dendrogram(WLDFresultLog,'WL_Log')
save_dendrogram(WLDFresultNonLog,'WL_notLog')

save_dendrogram(BLDFresultLog,'BL_Log')
save_dendrogram(BLDFresultNonLog,'BL_notLog')
save_dendrogram(DFfreq,'vanillaRelfreq')
save_dendrogram(DFfreqLog,'vanillaRelfreqLog')



# Extra - 3 time periods 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage

# Ensure 'Date' column is in datetime format
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

# Define the three time segments
periods = {
    "pre2014": merged_df[merged_df['Date'] < "2014-01-01"],
    "2014-2022_01": merged_df[(merged_df['Date'] >= "2014-01-01") & (merged_df['Date'] < "2022-02-01")],
    "2022_02onwards": merged_df[merged_df['Date'] >= "2022-02-01"]
}

# Iterate through each period, create a dendrogram, and save the figure.
for period_name, df_period in periods.items():
    # Pivot data: rows = language, columns = Date, values = Expression
    df_pivot = df_period.pivot(index='language', columns='Date', values='Expression')
    # Ensure date columns are sorted chronologically.
    df_pivot = df_pivot.sort_index(axis=1)
    # Handle missing data by filling forward then backward.
    df_pivot = df_pivot.fillna(method='ffill', axis=1).fillna(method='bfill', axis=1)
    
    # Check if there is enough data (at least two languages) to cluster
    if df_pivot.shape[0] < 2:
        print(f"Not enough data for the period: {period_name}")
        continue
    
    # Compute hierarchical clustering using Ward's method.
    Z = linkage(df_pivot, method='ward')
    
    # Create the dendrogram
    plt.figure(figsize=(9, 4))
    dendrogram(Z, labels=df_pivot.index, leaf_rotation=90)
    plt.title(f"Dendrogram of Languages based on Expression ({period_name})")
    plt.xlabel("Language")
    plt.ylabel("Distance")
    plt.tight_layout()
    
    # Save the dendrogram as a PNG file with a resolution of 300 dpi.
    plt.savefig(f"dendrogram_{period_name}.png", format="png", dpi=200)
    plt.show()
